# 02 - Dataset de demanda y modelo de aceptación

## Fase 3 -- Construcción del dataset de demanda

Unimos `accepted` (`acepto_prestamo = 1`) y `rejected` (`acepto_prestamo = 0`) en un único dataset, usando el mapeo de columnas documentado en `notebooks/01_eda.ipynb` e implementado en `src/preprocessing.build_demand_dataset`.

**Limitación metodológica (ya introducida en el README, se repite aquí porque es central para todo lo que sigue):** en `rejected` no siempre se distingue si fue el *cliente* quien rechazó una oferta o el *banco* quien rechazó la solicitud antes de llegar a ofertar nada. Phillips (2013) define `F̄_i(r)` como la fracción de solicitantes *que ya pasaron underwriting* que acepta la tasa `r`. Nuestra variable `acepto_prestamo` es un proxy razonable de eso -- toda fila de `accepted` sí pasó underwriting y aceptó; toda fila de `rejected` no llegó a tener un préstamo, sin más granularidad sobre la razón -- pero no es una medición perfecta. Se documenta y se sigue adelante porque es la mejor aproximación disponible en un dataset público.

Además, `int_rate` (la tasa) solo existe para `accepted`: un rechazo nunca llega a que se le cotice una tasa. Esa asimetría se maneja explícitamente en la Fase 5 (feature engineering del modelo de demanda), no se inventa un valor aquí.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import build_demand_dataset  # noqa: E402

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (9, 4)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

accepted_clean = pd.read_csv(PROCESSED_DIR / "accepted_clean.csv.gz", low_memory=False)
rejected_clean = pd.read_csv(PROCESSED_DIR / "rejected_clean.csv.gz", low_memory=False)

print(f"accepted_clean: {accepted_clean.shape}")
print(f"rejected_clean: {rejected_clean.shape}")

accepted_clean: (149996, 155)
rejected_clean: (149993, 14)


In [2]:
demand = build_demand_dataset(accepted_clean, rejected_clean)

print(f"demand: {demand.shape}")
print()
print(demand["acepto_prestamo"].value_counts(normalize=True))
print()
demand.head()

demand: (299989, 12)

acepto_prestamo
1    0.500005
0    0.499995
Name: proportion, dtype: float64



,acepto_prestamo,monto,dti,fico,risk_score,emp_length_years,purpose,zip_code,addr_state,int_rate,fecha_solicitud,source_dataset
0,1,16000.0,26.40,722.0,NaN,1.0,debt_consolidation,786xx,TX,12.88,Dec-2015,accepted
1,1,29900.0,21.77,717.0,NaN,10.0,debt_consolidation,219xx,MD,12.88,Dec-2015,accepted
2,1,20000.0,18.83,837.0,NaN,9.0,debt_consolidation,125xx,NY,5.32,Dec-2015,accepted
3,1,16000.0,23.35,682.0,NaN,0.0,debt_consolidation,880xx,NM,17.97,Dec-2015,accepted
4,1,25000.0,34.53,732.0,NaN,9.0,debt_consolidation,212xx,MD,13.99,Dec-2015,accepted


In [3]:
print("% de missing por columna en el dataset de demanda:")
print(demand.isna().mean().sort_values(ascending=False))

% de missing por columna en el dataset de demanda:
risk_score          0.833137
fico                0.499995
int_rate            0.499995
emp_length_years    0.049598
dti                 0.000987
monto               0.000000
acepto_prestamo     0.000000
purpose             0.000000
zip_code            0.000000
addr_state          0.000000
fecha_solicitud     0.000000
source_dataset      0.000000
dtype: float64


Como es esperable por construcción: `risk_score` falta en ~83% del dataset combinado (100% de `accepted` + el 66.6% de `rejected` que ya sabíamos que faltaba), `fico` e `int_rate` faltan en ~50% (exactamente las filas de `rejected`). Esto no es un error -- es la asimetría estructural entre ambos datasets, y el modelo de demanda de la Fase 5 se diseña sabiendo esto (features presentes en ambos datasets, tratamiento explícito de missing por modelo).

In [4]:
demand.to_csv(PROCESSED_DIR / "demand_dataset.csv.gz", index=False, compression="gzip")
print("Guardado:", PROCESSED_DIR / "demand_dataset.csv.gz")

Guardado: C:\Users\josue\credit-pricing-optimizer\data\processed\demand_dataset.csv.gz
